# 🏋️ Практикум: 100 задач с решениями

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justxor/math-for-ai-2026/blob/main/notebooks/practicum.ipynb)

Код из раздела [PRACTICE.md](https://github.com/justxor/math-for-ai-2026/blob/main/PRACTICE.md#practice). Запускайте ячейки сверху вниз: последующие используют переменные предыдущих. Теория, формулы и картинки — в тексте курса по ссылке.

In [ ]:
# Colab: всё нужное уже установлено. Локально: pip install -r requirements.txt
import numpy as np
import math
np.set_printoptions(precision=4, suppress=True)

### Задача 3. Степенной метод

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
def power_iteration(A, iters=200):
    v = np.random.randn(A.shape[0])
    for _ in range(iters):
        v = A @ v; v /= np.linalg.norm(v)
    return v @ A @ v, v           # отношение Рэлея
A = np.array([[4., 1], [2, 3]])
print(power_iteration(A)[0])       # 5.0

### Задача 4. Сжатие изображения через SVD

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
img = np.random.rand(512, 512)                    # подставьте реальное изображение в градациях серого
U, S, Vt = np.linalg.svd(img, full_matrices=False)
approx = lambda k: (U[:, :k] * S[:k]) @ Vt[:k]
energy = np.cumsum(S**2) / np.sum(S**2)           # доля «энергии» для выбора k

### Задача 5. Косинусная близость батчем

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
Q = np.random.randn(100, 384); D = np.random.randn(10_000, 384)
Qn = Q / np.linalg.norm(Q, axis=1, keepdims=True)
Dn = D / np.linalg.norm(D, axis=1, keepdims=True)
S = Qn @ Dn.T                                      # (100, 10000)
top3 = np.argpartition(-S, 3, axis=1)[:, :3]       # O(n) вместо полной сортировки
top3 = np.take_along_axis(top3, np.argsort(-np.take_along_axis(S, top3, 1), 1), 1)

### Задача 6. Проекционная матрица

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
X = np.random.randn(10, 3)
P = X @ np.linalg.solve(X.T @ X, X.T)
print(np.allclose(P @ P, P), np.allclose(P, P.T), round(np.trace(P), 6))   # True True 3.0

### Задача 8. Якобиан softmax

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
z = np.random.randn(4); p = np.exp(z) / np.exp(z).sum()
J = np.diag(p) - np.outer(p, p)
h = 1e-6
J_num = np.array([(np.exp(z + h*e)/np.exp(z + h*e).sum() - np.exp(z - h*e)/np.exp(z - h*e).sum()) / (2*h)
                  for e in np.eye(4)]).T
print(np.allclose(J, J_num, atol=1e-8), np.allclose(J.sum(0), 0))

### Задача 10. Мини-autograd

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
import math
class Value:
    def __init__(self, data, parents=(), backward=lambda: None):
        self.data, self.grad, self._parents, self._backward = data, 0.0, parents, backward
    def __add__(self, o):
        o = o if isinstance(o, Value) else Value(o)
        out = Value(self.data + o.data, (self, o))
        def bw(): self.grad += out.grad; o.grad += out.grad
        out._backward = bw; return out
    def __mul__(self, o):
        o = o if isinstance(o, Value) else Value(o)
        out = Value(self.data * o.data, (self, o))
        def bw(): self.grad += o.data * out.grad; o.grad += self.data * out.grad
        out._backward = bw; return out
    def tanh(self):
        t = math.tanh(self.data); out = Value(t, (self,))
        def bw(): self.grad += (1 - t**2) * out.grad
        out._backward = bw; return out
    def backward(self):
        order, seen = [], set()
        def topo(v):
            if v not in seen:
                seen.add(v); [topo(p) for p in v._parents]; order.append(v)
        topo(self); self.grad = 1.0
        for v in reversed(order): v._backward()

x, w, b = Value(0.5), Value(-1.2), Value(0.3)
y = (x * w + b).tanh(); y.backward()
print(y.data, w.grad)   # w.grad = (1 - tanh²) · x

### Задача 14. SGD vs GD экспериментально

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
rng = np.random.default_rng(0)
X = rng.normal(size=(10_000, 20)); w_true = rng.normal(size=20)
y = (rng.random(10_000) < 1 / (1 + np.exp(-X @ w_true))).astype(float)
sig = lambda z: 1 / (1 + np.exp(-z))
loss = lambda w: np.mean(np.logaddexp(0, X @ w) - y * (X @ w))

def train(bs, lr, epochs=20):
    w = np.zeros(20); hist = []
    for ep in range(epochs):
        idx = rng.permutation(len(y))
        for s in range(0, len(y), bs):
            b = idx[s:s + bs]
            w -= lr * X[b].T @ (sig(X[b] @ w) - y[b]) / len(b)
        hist.append(round(loss(w), 3))
    return hist
print("GD ", train(10_000, 1.0))
print("SGD", train(32, 0.1))

### Задача 15. Gradient clipping

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
def clip_by_global_norm(grads, max_norm):
    total = np.sqrt(sum((g**2).sum() for g in grads))
    scale = min(1.0, max_norm / (total + 1e-6))
    return [g * scale for g in grads], total

### Задача 20. Сэмплирование из softmax через Gumbel

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
z = np.array([1.0, 2.0, 0.5])
u = np.random.rand(200_000, 3)
samples = np.argmax(z - np.log(-np.log(u)), axis=1)
print(np.bincount(samples) / len(samples), np.exp(z) / np.exp(z).sum())   # совпадают

### Задача 23. Репараметризация

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
mu = np.array([1.0, -2.0]); Sigma = np.array([[2.0, 0.8], [0.8, 1.0]])
L = np.linalg.cholesky(Sigma)
x = mu + np.random.randn(100_000, 2) @ L.T
print(x.mean(0).round(2), np.cov(x.T).round(2))

### Задача 33. Логистическая регрессия с нуля

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
X, y = make_classification(2000, 10, random_state=0)
w, b, lam, lr = np.zeros(10), 0.0, 1e-2, 0.5
for _ in range(3000):
    p = 1 / (1 + np.exp(-(X @ w + b)))
    w -= lr * (X.T @ (p - y) / len(y) + lam * w)
    b -= lr * (p - y).mean()
sk = LogisticRegression(C=1 / (lam * len(y))).fit(X, y)   # C = 1/(λN) из-за разной нормировки
print(np.round(w[:4], 3), np.round(sk.coef_[0][:4], 3))   # близки

### Задача 35. Почему residual помогает — численно

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
import torch
def grad_norm(residual, depth=50, d=64):
    torch.manual_seed(0)
    layers = [torch.nn.Linear(d, d) for _ in range(depth)]
    x = torch.randn(32, d, requires_grad=True); h = x
    for l in layers:
        h = h + torch.tanh(l(h)) * 0.1 if residual else torch.tanh(l(h))
    h.sum().backward()
    return x.grad.norm().item()
print(grad_norm(False), grad_norm(True))   # без residual — на порядки меньше

### Задача 39. Speculative decoding

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
rng = np.random.default_rng(0)
p = np.array([0.5, 0.3, 0.2]); q = np.array([0.2, 0.2, 0.6])
resid = np.maximum(p - q, 0); resid /= resid.sum()
out = []
for _ in range(200_000):
    x = rng.choice(3, p=q)
    out.append(x if rng.random() < min(1, p[x] / q[x]) else rng.choice(3, p=resid))
print(np.bincount(out) / len(out))   # ≈ [0.5, 0.3, 0.2]

### Задача 40. Шаг диффузии

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
import torch
T = 1000
betas = torch.linspace(1e-4, 0.02, T); abar = torch.cumprod(1 - betas, 0)

def ddpm_loss(model, x0):
    t = torch.randint(0, T, (x0.shape[0],))
    eps = torch.randn_like(x0)
    a = abar[t].view(-1, *[1] * (x0.dim() - 1))
    xt = a.sqrt() * x0 + (1 - a).sqrt() * eps
    return torch.nn.functional.mse_loss(model(xt, t), eps)

dummy = lambda x, t: torch.zeros_like(x)
print(ddpm_loss(dummy, torch.randn(8, 3, 32, 32)))   # ≈ 1.0: предсказание «нулевого шума» даёт дисперсию ε

### Задача 53. Квантизация INT8

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
W = np.random.randn(256, 512).astype(np.float32)
W[3, 7] = 40.0                                        # выброс в канале 3
def quant(W, per_channel=True):
    m = np.abs(W).max(axis=1, keepdims=True) if per_channel else np.abs(W).max()
    s = m / 127
    return np.clip(np.round(W / s), -127, 127) * s
for pc in [False, True]:
    E = quant(W, pc) - W
    print("поканально" if pc else "на тензор ", np.abs(E).mean().round(4), np.abs(E[3]).mean().round(4))

### Задача 55. Policy gradient на бандите

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
rng = np.random.default_rng(0)
mu = np.array([1.0, 1.5, 2.0]) + 10                   # большие награды → большая дисперсия без baseline
def grad_samples(theta, baseline, n=2000):
    p = np.exp(theta - theta.max()); p /= p.sum()
    a = rng.choice(3, n, p=p); r = mu[a] + rng.normal(size=n)
    glogp = np.eye(3)[a] - p                          # ∇θ log softmax
    return glogp * (r - baseline)[:, None]
theta = np.zeros(3)
for b in [0.0, 11.5]:
    g = grad_samples(theta, b)
    print(f"baseline={b:5}: среднее {g.mean(0).round(3)}, дисперсия {g.var(0).sum():.2f}")

### Задача 57. Ранг обновлений при дообучении

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
d, r_true = 512, 8
dW = np.random.randn(d, r_true) @ np.random.randn(r_true, d) + 0.3 * np.random.randn(d, d)
S = np.linalg.svd(dW, compute_uv=False)
energy = np.cumsum(S**2) / np.sum(S**2)
print(np.searchsorted(energy, 0.9) + 1, "из", d)

### Задача 59. Calibration через temperature scaling

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
from scipy.optimize import minimize_scalar
rng = np.random.default_rng(0)
N, K = 5000, 10
y = rng.integers(0, K, N)
logits = rng.normal(size=(N, K)); logits[np.arange(N), y] += 1.5
logits *= 3                                            # искусственная переуверенность
def nll(T):
    z = logits / T; z = z - z.max(1, keepdims=True)
    return -(z[np.arange(N), y] - np.log(np.exp(z).sum(1))).mean()
T = minimize_scalar(nll, bounds=(0.1, 10), method="bounded").x
print(round(T, 2), nll(1.0).round(3), nll(T).round(3))   # T ≈ 2: NLL падает с 1.72 до 1.42

### Задача 61. Softmax, который иногда возвращает NaN

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
def softmax(z):
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)
p = softmax(np.array([[1000., 1001.], [1., 2.]]))
assert np.all(np.isfinite(p)) and np.allclose(p.sum(1), 1)

### Задача 62. Стандартизация по батчу

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
X = np.random.randn(256, 10) * np.arange(1, 11) + 3
X_norm = (X - X.mean(axis=0)) / X.std(axis=0)
assert np.allclose(X_norm.mean(0), 0, atol=1e-10) and np.allclose(X_norm.std(0), 1)

### Задача 63. Кросс-энтропия по неправильной оси

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
def cross_entropy(logits, y):
    z = logits - logits.max(1, keepdims=True)
    log_p = z - np.log(np.exp(z).sum(1, keepdims=True))
    return -log_p[np.arange(len(y)), y].mean()
logits = np.random.randn(32, 5); y = np.random.randint(0, 5, 32)
assert np.isclose(cross_entropy(logits, y), -np.mean(np.log(softmax(logits)[np.arange(32), y])))

### Задача 64. Утечка при нормировке

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
X = np.random.randn(500, 4)
X_tr, X_te = train_test_split(X, test_size=0.2, random_state=0)
sc = StandardScaler().fit(X_tr)
X_tr, X_te = sc.transform(X_tr), sc.transform(X_te)
assert np.allclose(X_tr.mean(0), 0, atol=1e-10)            # среднее теста не обязано быть 0

### Задача 65. Градиентный спуск, который уходит вверх

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
w = np.zeros(3); X = np.random.randn(100, 3); y = X @ np.array([1., -2., .5])
for _ in range(500):
    w -= 0.1 * X.T @ (X @ w - y) / len(y)
assert np.allclose(w, [1, -2, .5], atol=1e-3)

### Задача 66. Adam без коррекции смещения

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
m = v = 0
for t in range(1, 6):
    g = 1.0
    m = 0.9 * m + 0.1 * g; v = 0.999 * v + 0.001 * g**2
    step = (m / (1 - 0.9**t)) / (np.sqrt(v / (1 - 0.999**t)) + 1e-8)
    assert np.isclose(step, 1.0)

### Задача 67. Dropout на инференсе

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
def dropout(h, p=0.5, training=True):
    if not training or p == 0:
        return h
    mask = np.random.rand(*h.shape) > p
    return h * mask / (1 - p)
h = np.ones((1000, 1000))
assert abs(dropout(h).mean() - 1) < 0.01 and np.array_equal(dropout(h, training=False), h)

### Задача 68. Несмещённая дисперсия, которая смещена

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
rng = np.random.default_rng(0)
samples = rng.normal(0, 1, (200_000, 5))
print(samples.var(axis=1, ddof=0).mean().round(3), samples.var(axis=1, ddof=1).mean().round(3))   # 0.8 и 1.0

### Задача 69. Косинусное сходство без нормировки

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
def top_k(query, docs, k=3):
    d = docs / np.linalg.norm(docs, axis=1, keepdims=True)
    q = query / np.linalg.norm(query)
    return np.argsort(-(d @ q))[:k]
docs = np.array([[1., 0.], [10., 9.], [0.7, 0.1]])
print(np.argsort(-(docs @ np.array([1., 0.])))[:1], top_k(np.array([1., 0.]), docs, 1))   # [1] против [0]

### Задача 70. Accuracy 99% на несбалансированных данных

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
y_true = np.r_[np.zeros(990), np.ones(10)]; y_pred = np.zeros(1000)
tp = np.sum((y_pred == 1) & (y_true == 1))
recall = tp / y_true.sum()
print((y_pred == y_true).mean(), recall)      # 0.99 и 0.0

### Задача 81. Дерево решений (критерий Джини)

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

def gini(y):
    p = np.bincount(y, minlength=2) / len(y)
    return 1 - np.sum(p**2)

def best_split(X, y):
    best = (None, None, gini(y))                       # (признак, порог, взвешенный gini)
    for j in range(X.shape[1]):
        for t in np.unique(np.quantile(X[:, j], np.linspace(0.05, 0.95, 19))):
            left = X[:, j] <= t
            if left.all() or not left.any():
                continue
            g = (left.sum() * gini(y[left]) + (~left).sum() * gini(y[~left])) / len(y)
            if g < best[2]:
                best = (j, t, g)
    return best

def build(X, y, depth=0, max_depth=4):
    j, t, _ = best_split(X, y)
    if depth == max_depth or j is None or len(y) < 10:
        return np.bincount(y, minlength=2).argmax()     # лист: мажоритарный класс
    left = X[:, j] <= t
    return (j, t, build(X[left], y[left], depth + 1, max_depth), build(X[~left], y[~left], depth + 1, max_depth))

def predict_one(node, x):
    while isinstance(node, tuple):
        j, t, l, r = node
        node = l if x[j] <= t else r
    return node

X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)
tree = build(Xtr, ytr)
mine = np.mean([predict_one(tree, x) == t for x, t in zip(Xte, yte)])
ref = DecisionTreeClassifier(max_depth=4, random_state=0).fit(Xtr, ytr).score(Xte, yte)
print(round(mine, 3), round(ref, 3))                     # ≈ 0.92 и ≈ 0.95 — почти как у sklearn

### Задача 82. Градиентный бустинг на «пеньках»

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
def fit_stump(x, r):
    best = (np.inf, None, None, None)
    for t in np.quantile(x, np.linspace(0.02, 0.98, 49)):
        l = x <= t
        if l.all() or not l.any():
            continue
        pl, pr = r[l].mean(), r[~l].mean()
        err = np.sum((r[l] - pl) ** 2) + np.sum((r[~l] - pr) ** 2)
        if err < best[0]:
            best = (err, t, pl, pr)
    return best[1:]

rng = np.random.default_rng(0)
x = rng.uniform(0, 6, 400); y = np.sin(x) + 0.2 * rng.normal(size=400)
F = np.full_like(y, y.mean()); lr = 0.1; stumps = []
for m in range(200):
    t, pl, pr = fit_stump(x, y - F)                     # остатки = −∇ MSE
    F += lr * np.where(x <= t, pl, pr); stumps.append((t, pl, pr))
print(round(np.mean((y - F) ** 2), 4))                  # ≈ шум 0.04 — модель выучила sin

### Задача 83. k-means++

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
from sklearn.datasets import make_blobs
X, _ = make_blobs(1500, centers=8, cluster_std=0.8, random_state=1)

def kmeans(X, C, iters=30):
    for _ in range(iters):
        lab = ((X[:, None] - C[None]) ** 2).sum(-1).argmin(1)
        C = np.array([X[lab == j].mean(0) if (lab == j).any() else C[j] for j in range(len(C))])
    return ((X - C[lab]) ** 2).sum()

def init_pp(X, k, rng):
    C = [X[rng.integers(len(X))]]
    for _ in range(k - 1):
        d2 = ((X[:, None] - np.array(C)[None]) ** 2).sum(-1).min(1)
        C.append(X[rng.choice(len(X), p=d2 / d2.sum())])   # дальние точки — вероятнее
    return np.array(C)

rng = np.random.default_rng(0)
rand = [kmeans(X, X[rng.choice(len(X), 8, replace=False)]) for _ in range(20)]
pp = [kmeans(X, init_pp(X, 8, rng)) for _ in range(20)]
print(round(np.mean(rand)), round(np.mean(pp)))         # k-means++ в среднем лучше

### Задача 84. PCA на рукописных цифрах

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
from sklearn.datasets import load_digits
X = load_digits().data
Xc = X - X.mean(0)
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
ratio = np.cumsum(S**2) / np.sum(S**2)
k = np.searchsorted(ratio, 0.9) + 1
X_rec = Xc @ Vt[:k].T @ Vt[:k] + X.mean(0)
print(k, round(np.mean((X - X_rec) ** 2) / X.var(), 3))  # 21 компонента из 64, ошибка ≈ 5% дисперсии

### Задача 85. Наивный Байес для текстов

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
from collections import Counter
train = [("выиграй приз бесплатно сейчас", 1), ("бесплатно деньги приз", 1), ("срочно выиграй деньги", 1),
         ("встреча завтра в офисе", 0), ("отчёт по проекту завтра", 0), ("обсудим проект на встрече", 0)]
vocab = sorted({w for t, _ in train for w in t.split()})
counts = {c: Counter(w for t, y in train if y == c for w in t.split()) for c in (0, 1)}
prior = {c: np.mean([y == c for _, y in train]) for c in (0, 1)}

def log_posterior(text, c, alpha=1.0):
    total = sum(counts[c].values())
    return np.log(prior[c]) + sum(np.log((counts[c][w] + alpha) / (total + alpha * len(vocab)))
                                  for w in text.split() if w in vocab)

for text in ["бесплатно приз", "встреча по проекту"]:
    print(text, "→ спам" if log_posterior(text, 1) > log_posterior(text, 0) else "→ не спам")

### Задача 86. Токенизатор BPE

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
from collections import Counter
corpus = "низкий низкий ниже новый новейший новые низко".split()
words = Counter(tuple(w) + ("</w>",) for w in corpus)
merges = []
for _ in range(10):
    pairs = Counter()
    for w, f in words.items():
        for a, b in zip(w, w[1:]):
            pairs[(a, b)] += f
    if not pairs:
        break
    best = max(pairs, key=pairs.get); merges.append(best)
    new = Counter()
    for w, f in words.items():
        out, i = [], 0
        while i < len(w):
            if i < len(w) - 1 and (w[i], w[i + 1]) == best:
                out.append(w[i] + w[i + 1]); i += 2
            else:
                out.append(w[i]); i += 1
        new[tuple(out)] += f
    words = new
print(merges[:5])
print(list(words)[:3])     # частые куски («низк», «нов») стали отдельными токенами

### Задача 87. Биграммная языковая модель и perplexity

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
text = ("градиентный спуск обновляет веса модели шаг за шагом пока функция потерь не перестанет "
        "уменьшаться и модель не начнёт хорошо предсказывать новые данные ") * 20
chars = sorted(set(text)); idx = {c: i for i, c in enumerate(chars)}; V = len(chars)
split = int(len(text) * 0.9); tr, te = text[:split], text[split:]
N = np.ones((V, V))                                         # сглаживание Лапласа
for a, b in zip(tr, tr[1:]):
    N[idx[a], idx[b]] += 1
P = N / N.sum(1, keepdims=True)
nll = -np.mean([np.log(P[idx[a], idx[b]]) for a, b in zip(te, te[1:])])
print(V, round(np.exp(nll), 2))                             # perplexity ≈ 6 против 29 у случайной модели

### Задача 88. MLP на цифрах

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
X, y = load_digits(return_X_y=True); X = X / 16.0
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)
rng = np.random.default_rng(0)
W1 = rng.normal(0, np.sqrt(2 / 64), (64, 128)); b1 = np.zeros(128)
W2 = rng.normal(0, np.sqrt(1 / 128), (128, 10)); b2 = np.zeros(10)
for epoch in range(30):
    batches = rng.permutation(len(Xtr))[: len(Xtr) // 32 * 32].reshape(-1, 32)   # перемешанные батчи по 32
    for i in batches:
        xb, yb = Xtr[i], ytr[i]
        a1 = xb @ W1 + b1; h = np.maximum(0, a1); z = h @ W2 + b2
        p = np.exp(z - z.max(1, keepdims=True)); p /= p.sum(1, keepdims=True)
        dz = p.copy(); dz[np.arange(32), yb] -= 1; dz /= 32
        dW2 = h.T @ dz; db2 = dz.sum(0); dh = dz @ W2.T * (a1 > 0); dW1 = xb.T @ dh; db1 = dh.sum(0)
        for P_, G in [(W1, dW1), (b1, db1), (W2, dW2), (b2, db2)]:
            P_ -= 0.1 * G
pred = (np.maximum(0, Xte @ W1 + b1) @ W2 + b2).argmax(1)
print(round((pred == yte).mean(), 3))                       # ≈ 0.97

### Задача 89. Логистическая регрессия с L1 через проксимальный шаг

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
rng = np.random.default_rng(0)
X = rng.normal(size=(500, 20)); w_true = np.zeros(20); w_true[:3] = [2, -3, 1.5]
y = (rng.random(500) < 1 / (1 + np.exp(-X @ w_true))).astype(float)
w = np.zeros(20); lr, lam = 0.5, 0.05
for _ in range(500):
    g = X.T @ (1 / (1 + np.exp(-X @ w)) - y) / len(y)
    w = w - lr * g
    w = np.sign(w) * np.maximum(np.abs(w) - lr * lam, 0)   # проксимальный оператор L1
print(np.round(w, 2)); print("нулевых весов:", np.sum(w == 0))

### Задача 90. Сравнение оптимизаторов на логистической регрессии

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
rng = np.random.default_rng(1)
X = rng.normal(size=(1000, 10)) * np.logspace(0, 2, 10)       # масштабы признаков от 1 до 100
y = (X[:, 0] + 0.01 * X[:, -1] > 0).astype(float)
loss = lambda w: np.mean(np.logaddexp(0, X @ w) - y * (X @ w))
grad = lambda w: X.T @ (1 / (1 + np.exp(-np.clip(X @ w, -30, 30))) - y) / len(y)
res = {}
for name in ["SGD", "Momentum", "Adam"]:
    w = np.zeros(10); m = np.zeros(10); v = np.zeros(10)
    for t in range(1, 201):
        g = grad(w)
        if name == "SGD":
            w -= 1e-4 * g
        elif name == "Momentum":
            m = 0.9 * m + g; w -= 1e-4 * m
        else:
            m = 0.9 * m + 0.1 * g; v = 0.999 * v + 0.001 * g**2
            w -= 0.05 * (m / (1 - 0.9**t)) / (np.sqrt(v / (1 - 0.999**t)) + 1e-8)
    res[name] = round(loss(w), 4)
print(res)        # Adam заметно ниже: он выравнивает масштабы признаков

### Задача 91. Математическое ожидание максимума

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
print(np.random.rand(1_000_000, 2).max(1).mean())   # ≈ 0.667

### Задача 92. Монти Холл

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
rng = np.random.default_rng(0); n = 100_000
prize = rng.integers(0, 3, n); pick = rng.integers(0, 3, n)
print((prize == pick).mean(), (prize != pick).mean())   # остаться ≈ 0.33, сменить ≈ 0.67

### Задача 95. Градиент нормировки вектора

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
x = np.random.randn(5); n = np.linalg.norm(x)
J = (np.eye(5) - np.outer(x, x) / n**2) / n
h = 1e-6; J_num = np.array([((x + h * e) / np.linalg.norm(x + h * e) - (x - h * e) / np.linalg.norm(x - h * e)) / (2 * h) for e in np.eye(5)]).T
print(np.allclose(J, J_num, atol=1e-6))

### Задача 99. Почему $\sqrt{d}$, а не $d$

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
d = 128; q, k = np.random.randn(d, 1000), np.random.randn(d, 1000)
s = (q * k).sum(0)
print(round(s.std(), 1), round((s / np.sqrt(d)).std(), 2), round((s / d).std(), 3))   # ≈ 11, 1.0, 0.09